# Notebook 3 · Machine learning: can a computer learn the physics?

**Question:** can a model that has **never seen the equations of motion** predict the range of a projectile with air resistance, just from examples?

We'll compare three models:
1. **Linear regression** on the raw inputs (a baseline)
2. **Linear regression with a physics feature**: we give it the SUVAT range $v_0^2 \sin 2\theta / g$
3. **A neural network** on the raw inputs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from projectile import simulate, g

## 1. Load the data and add a physics feature

In [ ]:
df = pd.read_csv("data/projectiles.csv")
df["suvat_range"] = df["v0"]**2 * np.sin(2 * np.radians(df["angle_deg"])) / g
df.head()

## 2. Split into training and test sets

The model learns from the **training set** (80%). We then test it on the **test set** (20%), which it has never seen. This is like revising with past papers and then sitting a *different* exam. It checks that the model has actually learned the pattern rather than memorising the answers.

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Training examples: {len(train)},  test examples: {len(test)}")

raw_features = ["v0", "angle_deg", "b"]
target = "range"

## 3. How we measure performance

- **MAE (mean absolute error):** on average, how many metres out the prediction is. Lower is better.
- **$R^2$:** the fraction of the variation in the data that the model explains. $1$ is perfect and $0$ is useless.

In [ ]:
results = {}

def evaluate(name, model, features):
    predictions = model.predict(test[features])
    mae = mean_absolute_error(test[target], predictions)
    r2 = r2_score(test[target], predictions)
    results[name] = {"MAE (m)": mae, "R²": r2}
    print(f"{name}:  MAE = {mae:.2f} m,  R² = {r2:.4f}")
    return predictions

## 4. Model 1: linear regression (baseline)

This fits $\text{range} \approx w_1 v_0 + w_2\theta + w_3 b + c$. Projectile motion is **not** linear (range depends on $v_0^2$ and $\sin 2\theta$), so we expect this to do badly.

In [ ]:
linear = LinearRegression()
linear.fit(train[raw_features], train[target])
pred_linear = evaluate("Linear regression", linear, raw_features)

## 5. Model 2: linear regression with a physics feature

Now we use our physics knowledge to help the model by giving it the SUVAT range as an input. This is called **feature engineering**.

In [ ]:
physics_features = ["suvat_range", "b"]
physics_linear = LinearRegression()
physics_linear.fit(train[physics_features], train[target])
pred_physics = evaluate("Linear + physics feature", physics_linear, physics_features)

## 6. Model 3: a neural network

A neural network is made of layers of "neurons". Each neuron calculates a weighted sum of its inputs and passes it through a simple non-linear function. Training adjusts the weights using **gradient descent** (repeatedly nudging every weight in the direction that reduces the error, which uses differentiation and the chain rule). With enough neurons, a network can approximate almost any smooth function.

Our network has 3 inputs → 64 neurons → 64 neurons → 1 output.

`StandardScaler` rescales every input (and the output) to have mean 0 and standard deviation 1. Neural networks train much better when all the numbers are a similar size.

Training takes roughly **10–60 seconds**.

In [ ]:
neural_net = TransformedTargetRegressor(
    regressor=make_pipeline(
        StandardScaler(),
        MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=2000, random_state=0),
    ),
    transformer=StandardScaler(),
)
neural_net.fit(train[raw_features], train[target])
pred_nn = evaluate("Neural network", neural_net, raw_features)

## 7. Compare the models

In [ ]:
pd.DataFrame(results).T.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, (name, pred) in zip(axes, [("Linear regression", pred_linear),
                                   ("Linear + physics feature", pred_physics),
                                   ("Neural network", pred_nn)]):
    ax.scatter(test[target], pred, s=5, alpha=0.6)
    ax.plot([0, 260], [0, 260], "k--", lw=1)
    ax.set_title(name)
    ax.set_xlabel("true range (m)")
axes[0].set_ylabel("predicted range (m)")
plt.tight_layout()
plt.show()

Points on the dashed line are perfect predictions. The neural network should be far better than plain linear regression, which shows it has learned the non-linear physics just from examples.

## 8. Using the model: find the best launch angle

A trained model is very **fast** compared with running a simulation. Let's use it to find the best launch angle and check its answer against the real simulation.

In [ ]:
v0_test, b_test = 30.0, 0.01
angles_fine = np.arange(5, 85.5, 0.5)

query = pd.DataFrame({"v0": v0_test, "angle_deg": angles_fine, "b": b_test})
predicted = neural_net.predict(query)
simulated = np.array([simulate(v0_test, a, b_test)[1][-1, 0] for a in angles_fine])

plt.figure(figsize=(8, 4.5))
plt.plot(angles_fine, simulated, "k-", lw=3, alpha=0.3, label="RK4 simulation")
plt.plot(angles_fine, predicted, "r--", label="neural network")
plt.xlabel("launch angle (°)")
plt.ylabel("range (m)")
plt.title(f"v0 = {v0_test} m/s, b = {b_test}")
plt.legend()
plt.show()

print(f"Best angle (simulation):     {angles_fine[np.argmax(simulated)]}°")
print(f"Best angle (neural network): {angles_fine[np.argmax(predicted)]}°")

## 9. Where machine learning fails: extrapolation

Our training data only had speeds from **5 to 50 m/s**. What happens if we ask about faster launches?

In [ ]:
print(f"{'v0 (m/s)':>9} {'true range':>11} {'NN predicts':>12} {'error':>8}")
for v0_new in [20, 40, 50, 60, 80, 100]:
    true = simulate(v0_new, 40, 0.01)[1][-1, 0]
    guess = neural_net.predict(pd.DataFrame({"v0": [v0_new], "angle_deg": [40], "b": [0.01]}))[0]
    note = "  <- outside training data" if v0_new > 50 else ""
    print(f"{v0_new:>9} {true:>10.1f}m {guess:>11.1f}m {guess - true:>+7.1f}m{note}")

Inside the training range the network is very accurate, but **outside it the errors grow quickly**. The network has learned a pattern that fits the data, **not** Newton's laws. This is an important limitation to discuss in your evaluation, and it's why physics equations are still essential.

## 10. Conclusions and ideas to extend the project

**What we found:**
- RK4 is 4th order and Euler is 1st order (Notebook 1).
- With air resistance, the best launch angle is less than 45° (Notebook 2).
- A neural network can learn the range very accurately from examples alone, but it can't extrapolate beyond its training data.

**Extension ideas** (pick one or two to make the project your own):
1. **Predict other targets:** change `target` to `"max_height"` or `"flight_time"`.
2. **Network size:** try `hidden_layer_sizes=(4,)`, `(16,)`, `(64, 64)` and `(256, 256)`. Plot test error against size to show underfitting and overfitting.
3. **Less data:** how many examples does the network need? Train on 50, 100, 500 and 2400 and plot the error.
4. **The inverse problem:** given `v0`, `angle_deg` and `range`, predict `b`. This is how you could *measure* the drag on a real ball.
5. **Real experiment:** film a ball being thrown, track it with the free [Tracker](https://physlets.org/tracker/) software and compare with your model.
6. **More physics:** add wind, or spin (the Magnus force) to `derivatives()`.
7. **Speed test:** use `%timeit` to compare how long the simulation and the neural network take per prediction.